# COMMUN

### Imports et configuration

In [1]:
import os
import yaml
import pandas as pd
import numpy as np
import requests
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from pathlib import Path
from sqlalchemy.exc import SQLAlchemyError

### Charger variables d'environnement depuis .env

In [2]:
load_dotenv()

True

In [3]:
try:
    ROOT_DIR = Path(__file__).resolve().parents[1]
except NameError:
    ROOT_DIR = Path.cwd().parent

CONFIG_PATH = ROOT_DIR / "config.yml"
SQL_FILES_PATH = ROOT_DIR / "etl"
print(CONFIG_PATH)
print(SQL_FILES_PATH)

c:\Users\mabed\Documents\Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes\config.yml
c:\Users\mabed\Documents\Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes\etl


### Définir le chemin racine du projet (quel que soit le dossier courant)

In [4]:
def load_config(path):
    with open(path, "r", encoding="utf-8") as f:
        config = yaml.safe_load(f)
        
    for section, values in config.items():
        for key, val in values.items():
            if isinstance(val, str) and val.startswith("${"):
                env_var = val.strip("${}")
                config[section][key] = os.getenv(env_var)    
    return config

### Lecture du fichier config.yml

In [8]:
conf = load_config(CONFIG_PATH)
db_conf = conf['database']
#sql_file_conf = conf['sqlfile']

# AHMED

In [ ]:
def test_postgres_connection(db_config: dict, db_name: str | None = None) -> str:
    """
    Teste la connexion à une base PostgreSQL et retourne un message lisible.

    Args:
        db_config (dict): Dictionnaire contenant les informations de connexion :
            - user : nom d'utilisateur PostgreSQL
            - password : mot de passe
            - host : adresse du serveur
            - port : port PostgreSQL
            - db_default : base par défaut (souvent 'postgres')
        db_name (str | None): Nom de la base à tester. Si None, utilise db_config['db_default'].

    Returns:
        str: Message de succès avec la version PostgreSQL ou message d'erreur.
    """

    # Choix de la base : soit celle fournie, soit la base par défaut
    db_to_use = db_name or db_config["db_default"]

    # Création de l'URL de connexion PostgreSQL compatible SQLAlchemy
    db_url = (
        f"postgresql+psycopg2://{db_config['user']}:{db_config['password']}"
        f"@{db_config['host']}:{db_config['port']}/{db_to_use}"
    )

    try:
        # Création de l'engine SQLAlchemy
        engine = create_engine(db_url)

        # Ouverture de la connexion
        with engine.connect() as conn:
            # Exécution d'une requête pour récupérer la version PostgreSQL
            version = conn.execute(text("SELECT version();")).scalar()

            # Message de succès
            message = (
                f"Connexion réussie à la base '{db_to_use}'."
                f"Version PostgreSQL : {version}"
            )
            return message

    except SQLAlchemyError as e:
        # Gestion des erreurs de connexion
        message = (
            f"Erreur de connexion à la base '{db_to_use}'."
            f"{e}"
        )
        return message

    finally:
        # Fermeture propre de l'engine
        if 'engine' in locals():
            engine.dispose()

# execution
test_postgres_connection(db_conf)

In [ ]:
def create_database(db_config: dict, db_name: str):
    """
    Vérifie si une base PostgreSQL existe et la crée si nécessaire.

    Args:
        db_config (dict): Dictionnaire contenant les informations de connexion :
            - user : nom d'utilisateur PostgreSQL
            - password : mot de passe
            - host : adresse du serveur
            - port : port PostgreSQL
            - db_default : base par défaut utilisée pour se connecter initialement
        db_name (str): Nom de la base à créer ou vérifier.

    Returns:
        None
    """

    # Construction de l'URL de connexion sur la base par défaut (souvent 'postgres')
    db_url = (
        f"postgresql+psycopg2://{db_config['user']}:{db_config['password']}"
        f"@{db_config['host']}:{db_config['port']}/{db_config['db_default']}"
    )
    try:
        # Création de l'engine avec autocommit pour exécuter CREATE DATABASE
        engine = create_engine(db_url, isolation_level="AUTOCOMMIT")

        with engine.connect() as conn:
            # Vérifier si la base existe déjà
            result = conn.execute(
                text("SELECT 1 FROM pg_database WHERE datname = :dbname"),
                {"dbname": db_name}
            )
            exists = result.scalar()  # Récupère le premier résultat (1 si la base existe)

            if not exists:
                # Crée la base si elle n'existe pas
                conn.execute(text(f'CREATE DATABASE "{db_name}"'))
                print(f"Base '{db_name}' créée.")
            else:
                print(f"Base '{db_name}' existe déjà.")
    except SQLAlchemyError as e:
        # Gestion des erreurs SQLAlchemy
        print(f"Erreur lors de la vérification ou création de la base : {e}")

    finally:
        # Fermeture propre de l'engine
        if 'engine' in locals():
            engine.dispose()


# execution
create_database(db_conf, db_conf["db_accm"])

In [ ]:
def execute_sql_file(db_conf: dict, db_name: str, sql_file_path: str) -> str:

    db_to_use = db_name
    sql_file = Path(sql_file_path)
    if not sql_file.is_file():
        return f"Fichier SQL introuvable : {sql_file_path}"
    db_url = (
        f"postgresql+psycopg2://{db_conf['user']}:{db_conf['password']}"
        f"@{db_conf['host']}:{db_conf['port']}/{db_to_use}"
    )
    try:
        engine = create_engine(db_url, isolation_level="AUTOCOMMIT")

        # Lire le contenu du fichier SQL
        sql_commands = sql_file.read_text(encoding="utf-8")

        with engine.connect() as conn:
            conn.execute(text(sql_commands))
        return f"Fichier SQL '{sql_file_path}' exécuté avec succès sur la base '{db_to_use}'."
    except SQLAlchemyError as e:
        return f"Erreur lors de l'exécution du fichier SQL : {e}"
    finally:
        if 'engine' in locals():
            engine.dispose()
execute_sql_file(db_conf, db_conf["db_accm"], SQL_FILES_PATH/sql_file_conf["file_1"])

In [ ]:
def execute_sql_to_df(db_conf: dict, db_name: str, sql_file_path: str) -> pd.DataFrame:
    """
    Exécute une requête SQL depuis un fichier sur une base PostgreSQL et retourne le résultat sous forme de DataFrame.

    Args:
        db_conf (dict): Dictionnaire contenant les informations de connexion :
            - user : nom d'utilisateur PostgreSQL
            - password : mot de passe
            - host : adresse du serveur
            - port : port PostgreSQL
            - db_default : base par défaut (optionnelle)
        db_name (str): Nom de la base de données sur laquelle exécuter la requête.
        sql_file_path (str): Chemin vers le fichier SQL contenant la ou les requêtes.

    Returns:
        pd.DataFrame: DataFrame contenant le résultat de la requête si des colonnes existent,
                      sinon un DataFrame avec un message d'information.
    """
    # Convertit le chemin du fichier SQL en objet Path pour faciliter les manipulations
    sql_file = Path(sql_file_path)

    # Vérifie que le fichier SQL existe, sinon retourne un DataFrame avec message d'erreur
    if not sql_file.is_file():
        print(f"Fichier SQL introuvable : {sql_file_path}")
        return pd.DataFrame({"info_message": [f"Fichier SQL introuvable : {sql_file_path}"]})

    # Construction de l'URL de connexion PostgreSQL compatible SQLAlchemy
    connection_url = (
        f"postgresql+psycopg2://{db_conf['user']}:{db_conf['password']}"
        f"@{db_conf['host']}:{db_conf['port']}/{db_name}"
    )
    try:
        # Création de l'objet engine SQLAlchemy avec autocommit pour exécuter DDL si nécessaire
        engine = create_engine(connection_url, isolation_level="AUTOCOMMIT")

        # Lecture du contenu du fichier SQL
        sql_text = sql_file.read_text(encoding="utf-8")

        # Ouverture d'une connexion à la base de données
        with engine.connect() as conn:
            # Exécution de la requête SQL
            result = conn.execute(text(sql_text))
            
            # Si la requête renvoie des lignes (ex: SELECT), créer un DataFrame
            if result.returns_rows:
                df = pd.DataFrame(result.fetchall(), columns=result.keys())
            else:
                # Si la requête ne renvoie rien (ex: CREATE TABLE), renvoyer un message
                df = pd.DataFrame({"info_message": ["Requête exécutée avec succès, pas de résultat à afficher."]})
            
            return df

    except SQLAlchemyError as e:
        # Capture des erreurs SQLAlchemy et retour d'un DataFrame contenant l'erreur
        return pd.DataFrame({"info_message": [f"Erreur lors de l'exécution : {e}"]})

    finally:
        # Libération des ressources de l'engine pour fermer proprement la connexion
        if 'engine' in locals():
            engine.dispose()


## Exemple

df = execute_sql_to_df(db_conf, db_conf["db_accm"], SQL_FILES_PATH/sql_file_conf["file_1"])
df.head()

# ROMAIN

In [13]:
API_CONFIG = {
    'base_url' : 'https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/accidents-corporels-de-la-circulation-millesime/records',
    'limit_per_request' : 100,
    'max_records' : 1000,
    'timeout' : 30
}

print(f"API: {API_CONFIG['base_url'][:70]}...")

API: https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/acci...


In [9]:
def extraire_accidents_api(max_records=None):
    """
    Fonction pour extraire les accidents depuis l'API
    
    Paramètres:
        max_records (int): Le nombre maximum d'accidents à extraire. Si None, tous les accidents seront extraits.
    
    Return:
        list: Une liste de dictionnaires représentant les accidents extraîts.
    """

print("=" * 80)
print("EXTRACTION DES DONNÉES")
print("=" * 80)

all_records = []
offset = 0
limit = API_CONFIG['limit_per_request']

try:
    #première requête pour connaitre le total
    print("Récupération du nombre total...")
    response = requests.get(
        API_CONFIG['base_url'],
        params={'limit': 1},
        timeout=API_CONFIG['timeout']
    )
    response.raise_for_status()
    data = response.json()
    total_count = data.get('total_count', 0)
    
    print(f"Total disponible: {total_count:,} enregistrements")
    print(data)       
except requests.RequestException as e:
    print(f"\n✗ Erreur lors de l'extraction: {e}")
    raise

EXTRACTION DES DONNÉES
Récupération du nombre total...
Total disponible: 475,911 enregistrements
{'total_count': 475911, 'results': [{'num_acc': '201700009715', 'datetime': '2017-05-28T16:50:00+00:00', 'nom_com': None, 'an': '2017', 'mois': '05', 'jour': '28', 'hrmn': '18:50', 'lum': 'Plein jour', 'agg': 'En agglomération', 'int': '3', 'atm': 'Normale', 'col': 'Deux véhicules – par le coté', 'dep': '13', 'com': '055', 'insee': '13055', 'adr': '6 Av Alexandre  Ansaldi', 'lat': '4333582', 'long': '0539866', 'code_postal': None, 'num': '6', 'coordonnees': {'lon': 2.911777, 'lat': 42.686216}, 'pr': None, 'surf': 'normale', 'v1': None, 'circ': 'Bidirectionnelle', 'vosp': None, 'env1': '00', 'voie': '4', 'larrout': 120, 'v2': None, 'lartpc': 25, 'nbv': 4, 'catr': 'Route Départementale', 'pr1': None, 'plan': 'Partie rectiligne', 'prof': 'Plat', 'infra': None, 'situ': 'Sur chaussée', 'an_nais': ['1998', '1966'], 'sexe': ['Masculin', 'Masculin'], 'actp': ['Se déplaçant', 'Se déplaçant'], 'grav'

In [9]:
"""Téléchargement minimaliste du dataset accidents corporels depuis OpenDataSoft.

Version simplifiée sans retry, sans barre de progression, sans validation.
Télécharge le CSV par chunks et le sauvegarde dans data/accidents_corporels_millesime.csv

Usage:
    python scripts/sauvegarde_csv_api_v1.py

Source:
    https://public.opendatasoft.com - Dataset accidents corporels de la circulation
"""

import sys
from dataclasses import dataclass
from pathlib import Path
import requests



@dataclass(frozen=True)
class Config:
    """Configuration du téléchargement."""
    
    base_url: str = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets"
    dataset_id: str = "accidents-corporels-de-la-circulation-millesime"
    output_dir: str = "data"
    output_file: str = "accidents_corporels_millesime.csv"
    delimiter: str = ","
    chunk_size: int = 65536 #on lit 64KB par 64KB pour ne pas que Lounes voit la RAM de son pc bruler
    timeout: int = 30


def build_url(config: Config) -> str:
    """Construit l'URL de téléchargement."""
    return f"{config.base_url}/{config.dataset_id}/exports/csv?delimiter={config.delimiter}"


def resolve_path(config: Config) -> Path:
    """Détermine le chemin de sortie."""
    #script_dir = ROOT_DIR
    return ROOT_DIR / config.output_dir / config.output_file


def download_csv(url: str, destination: Path, config: Config) -> None:
    """Télécharge le CSV par chunks."""
    print(f"Téléchargement depuis OpenDataSoft...")

    response = requests.get(url, stream=True, timeout=config.timeout)
    response.raise_for_status()

    destination.parent.mkdir(parents=True, exist_ok=True)

    with response, open(destination, "wb") as handle:
        for chunk in response.iter_content(chunk_size=config.chunk_size):
            if chunk:
                handle.write(chunk)

    print(f"Fichier sauvegardé: {destination}")


def main() -> None:
    """Point d'entrée principal."""
    config = Config()
    url = build_url(config)
    path = resolve_path(config)

    download_csv(url, path, config)

    print("Téléchargement terminé")


if __name__ == "__main__":
    main()


Téléchargement depuis OpenDataSoft...
Fichier sauvegardé: c:\Users\mabed\Documents\Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes\data\accidents_corporels_millesime.csv
Téléchargement terminé


In [8]:
# Import du script avec chemin absolu
import sys

# Chemin absolu vers le dossier scripts
scripts_dir = ROOT_DIR / 'etl'

# Ajouter au path Python
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

# Import
from sauvegarde_csv_api import download_accidents

# Téléchargement
stats = download_accidents(
    output=ROOT_DIR / 'data/accidents_corporels_millesime.csv',
    delimiter=';',
    show_progress=True,
    retries=3,
    #limit = 1000,
    #where ="an=2017 AND dep='60'"
)

print(f"✅ Téléchargement réussi !")
print(f"   - {stats.record_count:,} accidents")
print(f"   - {stats.column_count} colonnes")
print(f"   - {stats.size_mb:.2f} MB")


🚗 TÉLÉCHARGEMENT DATASET ACCIDENTS CORPORELS
📍 Source: OpenDataSoft
📅 Période: 2012-2019
📄 Fichier: /home/samchaka/simplon/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/data/accidents_corporels_millesime.csv
🔤 Séparateur: ','

⚠️  Module 'tqdm' non installé - pas de barre de progression détaillée
   Installez-le avec: pip install tqdm

🌐 Connexion à OpenDataSoft...
📥 Téléchargement (taille inconnue)
   Téléchargé: 354,434.8 KB
✅ Fichier sauvegardé: /home/samchaka/simplon/Projet-1-Simplon-Data-Engineer-2025---Mighty-Mosquitoes/data/accidents_corporels_millesime.csv
⏱️  Temps de téléchargement: 321.3 secondes

📊 RÉSUMÉ DU TÉLÉCHARGEMENT
✅ Fichier valide
   Taille: 346.13 MB (362,941,194 bytes)
   Lignes: 475,912 (incluant l'en-tête)
   Records: 475,911 accidents
   Colonnes: 69

📋 Premières colonnes:
   1. num_acc
   2. datetime
   3. nom_com
   4. an
   5. mois
   6. jour
   7. hrmn
   8. lum
   9. agg
   10. int
   ... et 59 autres colonnes

✅ Téléchargement réussi !
   - 47

# LOUNES

In [10]:
# Importation des données source dans un DataFrame pandas

df_source = pd.read_csv(ROOT_DIR / 'data/accidents_corporels_millesime.csv', delimiter=',', low_memory=False)
df_source.head()

,num_acc,datetime,nom_com,an,mois,jour,hrmn,lum,agg,int,...,year_georef,com_name,dep_code,dep_name,epci_code,epci_name,reg_code,reg_name,com_arm_name,com_code
0,201900020750,2019-01-29T15:45:00+00:00,Corbeil-essonnes,2019,1,29,16:45,Plein jour,En agglomération,2,...,2019,Corbeil-Essonnes,91.0,Essonne,200059228.0,CA Grand Paris Sud Seine Essonne Sénart,11.0,Île-de-France,Corbeil-Essonnes,91174.0
1,201900020796,2019-10-07T17:30:00+00:00,Istres,2019,10,7,19:30,Nuit sans éclairage public,Hors agglomération,1,...,2019,Istres,13.0,Bouches-du-Rhône,200054807.0,Métropole d'Aix-Marseille-Provence,93.0,Provence-Alpes-Côte d'Azur,Istres,13047.0
2,201900020869,2019-10-13T13:46:00+00:00,Saint-laurent-du-pont,2019,10,13,15:46,Plein jour,En agglomération,1,...,2019,Saint-Laurent-du-Pont,38.0,Isère,200040111.0,CC Coeur de Chartreuse,84.0,Auvergne-Rhône-Alpes,Saint-Laurent-du-Pont,38412.0
3,201900021309,2019-04-17T14:30:00+00:00,Livry-gargan,2019,4,17,16:30,Plein jour,En agglomération,1,...,2019,Livry-Gargan,93.0,Seine-Saint-Denis,200054781.0,Métropole du Grand Paris,11.0,Île-de-France,Livry-Gargan,93046.0
4,201900018753,2019-12-25T17:10:00+00:00,Gennevilliers,2019,12,25,18:10,Nuit sans éclairage public,Hors agglomération,1,...,2019,Gennevilliers,92.0,Hauts-de-Seine,200054781.0,Métropole du Grand Paris,11.0,Île-de-France,Gennevilliers,92036.0


In [11]:
# Exploration de la donnée source csv

df_source.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 69 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   num_acc       475911 non-null  int64  
 1   datetime      475911 non-null  object 
 2   nom_com       449206 non-null  object 
 3   an            475911 non-null  int64  
 4   mois          475911 non-null  int64  
 5   jour          475911 non-null  int64  
 6   hrmn          475911 non-null  object 
 7   lum           475911 non-null  object 
 8   agg           475911 non-null  object 
 9   int           475911 non-null  int64  
 10  atm           475861 non-null  object 
 11  col           475901 non-null  object 
 12  dep           475911 non-null  object 
 13  com           475911 non-null  object 
 14  insee         475296 non-null  float64
 15  adr           426655 non-null  object 
 16  lat           300785 non-null  object 
 17  long          300785 non-null  object 
 18  code

In [12]:
"""
Préparation des données : nettoyage, transformation

Objectif : diviser le csv en 5 tables distinctes (accident, vehicule, usager, lieux, date)
Chaque table sera dans une dataframe pandas distincte

1ere étape : premier petit nettoyage et uniformisation des données
2eme étape : séparation en 5 dataframes
3ème étape : explosion des colonnes multi-valeurs pour les dataframes vehicule et usager
4eme étape : nettoyage spécifique à chaque dataframe
5ème étape : mapping des valeurs catégorielles pour chaque dataframe
6ème étape : tests de validation des données
7ème étape : export des dataframes nettoyés et validés dans postgres
"""

# Premier nettoyage et uniformisation des données
df_source.columns = df_source.columns.str.lower().str.strip()
df_source["datetime"] = pd.to_datetime(df_source["datetime"], errors="coerce")
df_source.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 69 columns):
 #   Column        Non-Null Count   Dtype              
---  ------        --------------   -----              
 0   num_acc       475911 non-null  int64              
 1   datetime      475911 non-null  datetime64[ns, UTC]
 2   nom_com       449206 non-null  object             
 3   an            475911 non-null  int64              
 4   mois          475911 non-null  int64              
 5   jour          475911 non-null  int64              
 6   hrmn          475911 non-null  object             
 7   lum           475911 non-null  object             
 8   agg           475911 non-null  object             
 9   int           475911 non-null  int64              
 10  atm           475861 non-null  object             
 11  col           475901 non-null  object             
 12  dep           475911 non-null  object             
 13  com           475911 non-null  object       

In [13]:
# Séparation en 5 dataframes
# 1. Accidents
df_accidents = df_source[[
    'num_acc', 
    'lum', 
    'agg',
    'int',
    'atm',
    'adr',
    'col',
    'circ',
    'plan',
    'prof',
    'surf',
    'infra',
    'situ',
    'year_georef'
]].copy()

# df_accidents.head()
df_accidents.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 14 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   num_acc      475911 non-null  int64 
 1   lum          475911 non-null  object
 2   agg          475911 non-null  object
 3   int          475911 non-null  int64 
 4   atm          475861 non-null  object
 5   adr          426655 non-null  object
 6   col          475901 non-null  object
 7   circ         450504 non-null  object
 8   plan         441487 non-null  object
 9   prof         447014 non-null  object
 10  surf         459780 non-null  object
 11  infra        54500 non-null   object
 12  situ         449540 non-null  object
 13  year_georef  475911 non-null  int64 
dtypes: int64(3), object(11)
memory usage: 50.8+ MB


In [14]:
# Tests de validation dataframe df_accidents

# Vérification des doublons sur 'num_acc
duplicates_accidents = df_accidents.duplicated(subset=['num_acc']).sum()
print(f"Doublons dans df_accidents sur 'num_acc': {duplicates_accidents} \n")

# Vérification des valeurs uniques sur les catégories
for col in ['lum', 'agg', 'int', 'atm', 'col', 'circ', 'plan', 'prof', 'surf', 'infra', 'situ']:
    unique_values = df_accidents[col].unique()
    print(f"\n Valeurs uniques dans '{col}': {unique_values}")

# Compter le nombre de 'nan' et '-1' dans chaque colonne catégorielle
for col in ['lum', 'agg', 'int', 'atm', 'col', 'circ', 'plan', 'prof', 'surf', 'infra', 'situ']:
    nan_count = df_accidents[col].isna().sum()
    neg_one_count = (df_accidents[col] == '-1').sum()
    print(f"\n Colonne '{col}': NaN = {nan_count}, -1 = {neg_one_count}")


Doublons dans df_accidents sur 'num_acc': 0 


 Valeurs uniques dans 'lum': ['Plein jour' 'Nuit sans éclairage public'
 'Nuit avec éclairage public non allumé'
 'Nuit avec éclairage public allumé' 'Crépuscule ou aube']

 Valeurs uniques dans 'agg': ['En agglomération' 'Hors agglomération']

 Valeurs uniques dans 'int': [2 1 9 6 3 7 4 5 0 8]

 Valeurs uniques dans 'atm': ['Normale' 'Temps couvert' 'Temps éblouissant' 'Pluie légère'
 'Pluie forte' 'Autre' 'Vent fort - tempête' 'Brouillard - fumée'
 'Neige - grêle' nan '-1']

 Valeurs uniques dans 'col': ['Deux véhicules – par le coté' 'Sans collision'
 'Deux véhicules - frontale' 'Deux véhicules – par l’arrière'
 'Autre collision' 'Trois véhicules et plus - collisions multiples'
 'Trois véhicules et plus – en chaîne' nan '-1']

 Valeurs uniques dans 'circ': ['Bidirectionnelle' '-1' 'A sens unique' 'A chaussées séparées' nan
 'Avec voies d’affectation variable']

 Valeurs uniques dans 'plan': ['Partie rectiligne' nan 'En courbe à droite' 

In [15]:
# Transformation des valeurs '-1' en NaN pour les colonnes catégorielles
for col in ['lum', 'agg', 'int', 'atm', 'col', 'circ', 'plan', 'prof', 'surf', 'infra', 'situ']:
    df_accidents[col] = df_accidents[col].replace('-1', np.nan)

In [16]:
# Nettoyer "infra" et "situ" car ont des numéros écris en type object et des catégories mélangées. Transformer les numéros en NaN
df_accidents['infra'] = df_accidents['infra'].apply(lambda x: np.nan if str(x).isdigit() else x)
df_accidents['situ'] = df_accidents['situ'].apply(lambda x: np.nan if str(x).isdigit() else x)

In [17]:
# La dataframe df_accidents est prête pour export. Mapping catégories à réaliser.



In [18]:
# Séparation en 5 dataframes
# 2. Lieux
df_lieux = df_source[[
    'com_code',
    'com_name',
    'dep_code',
    'dep_name',
    'reg_code', 
    'reg_name', 
    'epci_code', 
    'epci_name',
    'lat', 
    'long', 
    'catr', 
    'v1', 
    'voie', 
    'v2', 
    'nbv', 
    'vosp', 
    'pr', 
    'pr1', 
    'lartpc', 
    'larrout', 
    'num_acc'
]].copy()

# df_lieux.head()
df_lieux.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 21 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   com_code   475296 non-null  float64
 1   com_name   464648 non-null  object 
 2   dep_code   464648 non-null  float64
 3   dep_name   464648 non-null  object 
 4   reg_code   464648 non-null  float64
 5   reg_name   464648 non-null  object 
 6   epci_code  426190 non-null  float64
 7   epci_name  426190 non-null  object 
 8   lat        300785 non-null  object 
 9   long       300785 non-null  object 
 10  catr       475911 non-null  object 
 11  v1         50267 non-null   float64
 12  voie       410712 non-null  object 
 13  v2         19498 non-null   object 
 14  nbv        474103 non-null  float64
 15  vosp       32080 non-null   object 
 16  pr         259060 non-null  object 
 17  pr1        250140 non-null  float64
 18  lartpc     362782 non-null  float64
 19  larrout    364392 non-n

In [19]:
# Vérification des doublons sur 'num_acc
duplicates_lieux = df_lieux.duplicated(subset=['num_acc']).sum()
print(f"Doublons dans df_lieux sur 'num_acc': {duplicates_lieux} \n")

# Vérification des catégories
# df_lieux['catr'].value_counts(dropna=False)

# Notes nettoyage : 
# drop 'voie', drop 'V1', drop 'V2', drop 'pr', drop 'pr1', drop 'lartpc', drop 'larrout, drop 'epci_code', drop 'epci_name', drop 'lat', drop 'long
# numéros à mettre en NaN dans 'catr'
# nbv > 6 ou nbv = -1 ou nbv = 0 --> NaN (explication : deux fois 3 voies sur autoroute est le max selon nous soit 6 voies)
# vosp = -1 --> NaN

# Drop des colonnes inutiles
df_lieux = df_lieux.drop(columns=['voie', 'v1', 'v2', 'pr', 'pr1', 'lartpc', 'larrout', 'epci_code', 'epci_name', 'lat', 'long'])


Doublons dans df_lieux sur 'num_acc': 0 



In [20]:
# Nettoyer "catr"
df_lieux['catr'] = df_lieux['catr'].apply(lambda x: 'autre' if str(x).isdigit() else x)

df_lieux['catr'].value_counts(dropna=False)

catr
Voie Communale                                            230977
Route Départementale                                      161026
Autoroute                                                  41951
Route Nationale                                            30360
autre                                                       7492
Parc de stationnement ouvert à la circulation publique      3334
Hors réseau public                                           771
Name: count, dtype: int64

In [21]:
# Nettoyer 'nbv'
# nbv > 6 ou nbv = -1 ou nbv = 0 --> NaN (explication : deux fois 3 voies sur autoroute est le max selon nous soit 6 voies). values are float.
df_lieux['nbv'] = df_lieux['nbv'].apply(lambda x: np.nan if (x > 6.0 or x == 0.0) or x == -1.0 else x)

# transformer le dtype en int
df_lieux['nbv'] = df_lieux['nbv'].astype('Int64')

In [22]:
# Nettoyer 'vosp'
# vosp = -1 --> NaN
df_lieux['vosp'] = df_lieux['vosp'].apply(lambda x: np.nan if x == '-1' else x)
df_lieux['vosp'].value_counts(dropna=False)

vosp
NaN                444500
Voie réservée       13686
Piste cyclable      11258
Banque cyclable      6467
Name: count, dtype: int64

In [23]:
# Last check df_lieux
df_lieux.info()

# df_lieux est prêt pour export dans postgres

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 10 columns):
 #   Column    Non-Null Count   Dtype  
---  ------    --------------   -----  
 0   com_code  475296 non-null  float64
 1   com_name  464648 non-null  object 
 2   dep_code  464648 non-null  float64
 3   dep_name  464648 non-null  object 
 4   reg_code  464648 non-null  float64
 5   reg_name  464648 non-null  object 
 6   catr      475911 non-null  object 
 7   nbv       424073 non-null  Int64  
 8   vosp      31411 non-null   object 
 9   num_acc   475911 non-null  int64  
dtypes: Int64(1), float64(3), int64(1), object(5)
memory usage: 36.8+ MB


In [24]:
# Séparation en 5 dataframes
# 3. Date_accident
df_date_accident = df_source[[
    'datetime', 
    'an', 
    'mois', 
    'jour', 
    'hrmn', 
    'num_acc'
]].copy()

# df_date_accident.head()
df_date_accident.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 6 columns):
 #   Column    Non-Null Count   Dtype              
---  ------    --------------   -----              
 0   datetime  475911 non-null  datetime64[ns, UTC]
 1   an        475911 non-null  int64              
 2   mois      475911 non-null  int64              
 3   jour      475911 non-null  int64              
 4   hrmn      475911 non-null  object             
 5   num_acc   475911 non-null  int64              
dtypes: datetime64[ns, UTC](1), int64(4), object(1)
memory usage: 21.8+ MB


In [25]:
# Vérification des colonnes 'an', 'mois', 'jour' pour éviter des aberrations
df_date_accident[['an', 'mois', 'jour']].describe()

# Vérification et transformation du dtype de la colonne 'hrmn' (exemple : dtype object "16:45") en datetime object "HH:MM"
df_date_accident['hrmn'] = pd.to_datetime(df_date_accident['hrmn'], format='%H:%M', errors='coerce').dt.time

In [26]:
# Last check for df_date_accident
df_date_accident.info()

# df_date_accident est prêt pour export dans postgres

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 6 columns):
 #   Column    Non-Null Count   Dtype              
---  ------    --------------   -----              
 0   datetime  475911 non-null  datetime64[ns, UTC]
 1   an        475911 non-null  int64              
 2   mois      475911 non-null  int64              
 3   jour      475911 non-null  int64              
 4   hrmn      475911 non-null  object             
 5   num_acc   475911 non-null  int64              
dtypes: datetime64[ns, UTC](1), int64(4), object(1)
memory usage: 21.8+ MB


In [27]:
# Séparation en 5 dataframes
# 4. Vehicules
df_vehicules = df_source[[
    'num_veh', 
    'catv', 
    'choc', 
    'senc', 
    'obs', 
    'obsm', 
    'occutc', 
    'manv', 
    'num_acc'
]].copy()

df_vehicules.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 475911 entries, 0 to 475910
Data columns (total 9 columns):
 #   Column   Non-Null Count   Dtype 
---  ------   --------------   ----- 
 0   num_veh  475911 non-null  object
 1   catv     475911 non-null  object
 2   choc     446407 non-null  object
 3   senc     209666 non-null  object
 4   obs      101364 non-null  object
 5   obsm     369261 non-null  object
 6   occutc   6015 non-null    object
 7   manv     441491 non-null  object
 8   num_acc  475911 non-null  int64 
dtypes: int64(1), object(8)
memory usage: 32.7+ MB


In [28]:
df_vehicules.head(20)

,num_veh,catv,choc,senc,obs,obsm,occutc,manv,num_acc
0,"B01,A01","Cyclomoteur <50cm3,VL seul",Avant,"3,3",NaN,Véhicule,NaN,Traversant la chaussée,201900020750
1,A01,VU seul 1,NaN,3,NaN,NaN,NaN,Même sens,201900020796
2,"B01,A01","VL seul,VL seul","Avant gauche,Avant",PK ou PR ou numéro d’adresse postale croissant...,NaN,"Véhicule,Véhicule",NaN,"Sans changement de direction,Déporté A gauche",201900020869
3,"B01,A01","43,VL seul",Avant,PK ou PR ou numéro d’adresse postale croissant...,NaN,"Véhicule,Véhicule",NaN,"Sans changement de direction,Manœuvre de stati...",201900021309
4,"C01,B01,A01","VL seul,VL seul,VL seul","Arrière,Arrière,Avant",PK ou PR ou numéro d’adresse postale décroissa...,NaN,"Véhicule,Véhicule,Véhicule",NaN,"Même sens,Même sens,Même sens",201900018753
5,"A01,B01","VL seul,VL seul","Avant,Côté gauche","3,3",NaN,"Véhicule,Véhicule",NaN,"Tournant A gauche,Sans changement de direction",201900037513
6,"C01,B01,A01","VL seul,VL seul,VL seul","Côté gauche,Côté gauche,Avant",NaN,NaN,"Véhicule,Véhicule,Véhicule",NaN,"Sans changement de direction,Sans changement d...",201400048899
7,A01,Motocyclette > 125 cm3,Avant,NaN,NaN,Autre,NaN,Entre 2 files,201400049030
8,A01,Autre véhicule,Avant,PK ou PR ou numéro d’adresse postale croissant,NaN,Piéton,NaN,Sans changement de direction,201700041615
9,"B01,A01","Engin spécial,VL seul","Avant,Arrière",PK ou PR ou numéro d’adresse postale croissant...,NaN,"Véhicule,Véhicule",NaN,"Sans changement de direction,Arrêté (hors stat...",201700042289


In [29]:
# Transformer les colonnes qui contiennent plusieurs valeurs séparées par des virgules en listes
veh_cols = ['num_veh', 'catv', 'choc', 'senc', 'obs', 'obsm', 'occutc', 'manv']
for col in veh_cols:
    df_vehicules[col] = df_vehicules[col].astype(str).apply(lambda x: x.split(',') if ',' in x else [x])

df_vehicules.head(20)

,num_veh,catv,choc,senc,obs,obsm,occutc,manv,num_acc
0,"[B01, A01]","[Cyclomoteur <50cm3, VL seul]",[Avant],"[3, 3]",[nan],[Véhicule],[nan],[Traversant la chaussée],201900020750
1,[A01],[VU seul 1],[nan],[3],[nan],[nan],[nan],[Même sens],201900020796
2,"[B01, A01]","[VL seul, VL seul]","[Avant gauche, Avant]",[PK ou PR ou numéro d’adresse postale croissan...,[nan],"[Véhicule, Véhicule]",[nan],"[Sans changement de direction, Déporté A gauche]",201900020869
3,"[B01, A01]","[43, VL seul]",[Avant],[PK ou PR ou numéro d’adresse postale croissan...,[nan],"[Véhicule, Véhicule]",[nan],"[Sans changement de direction, Manœuvre de sta...",201900021309
4,"[C01, B01, A01]","[VL seul, VL seul, VL seul]","[Arrière, Arrière, Avant]",[PK ou PR ou numéro d’adresse postale décroiss...,[nan],"[Véhicule, Véhicule, Véhicule]",[nan],"[Même sens, Même sens, Même sens]",201900018753
5,"[A01, B01]","[VL seul, VL seul]","[Avant, Côté gauche]","[3, 3]",[nan],"[Véhicule, Véhicule]",[nan],"[Tournant A gauche, Sans changement de direction]",201900037513
6,"[C01, B01, A01]","[VL seul, VL seul, VL seul]","[Côté gauche, Côté gauche, Avant]",[nan],[nan],"[Véhicule, Véhicule, Véhicule]",[nan],"[Sans changement de direction, Sans changement...",201400048899
7,[A01],[Motocyclette > 125 cm3],[Avant],[nan],[nan],[Autre],[nan],[Entre 2 files],201400049030
8,[A01],[Autre véhicule],[Avant],[PK ou PR ou numéro d’adresse postale croissant],[nan],[Piéton],[nan],[Sans changement de direction],201700041615
9,"[B01, A01]","[Engin spécial, VL seul]","[Avant, Arrière]",[PK ou PR ou numéro d’adresse postale croissan...,[nan],"[Véhicule, Véhicule]",[nan],"[Sans changement de direction, Arrêté (hors st...",201700042289


In [30]:
# 1️⃣ Compute length of each list cell for the vehicule columns
list_lengths_df = df_vehicules[veh_cols].map(
    lambda x: len(x) if isinstance(x, list) else 1
)

# 2️⃣ Compute how many distinct lengths there are per row
df_vehicules['nunique_lengths'] = list_lengths_df.nunique(axis=1)

# 3️⃣ (Optional) Store the max length — useful for padding later
df_vehicules['max_length'] = list_lengths_df.max(axis=1)

# 4️⃣ Identify misaligned rows (drifting)
misaligned_rows = df_vehicules[df_vehicules['nunique_lengths'] > 1]

print(f"⚠️ {len(misaligned_rows)} rows with misaligned list lengths detected.")

⚠️ 290537 rows with misaligned list lengths detected.


In [31]:
from itertools import zip_longest

def pad_lists(row):
    max_len = row['max_length']
    for col in veh_cols:
        vals = row[col] if isinstance(row[col], list) else [row[col]]
        row[col] = (vals + [None] * (max_len - len(vals)))[:max_len]
    return row

df_vehicules = df_vehicules.apply(pad_lists, axis=1)

In [32]:
# Retirer les colonnes nunique et max
df_vehicules = df_vehicules.drop(columns=['nunique_lengths', 'max_length'])

df_vehicules.head(10)

,num_veh,catv,choc,senc,obs,obsm,occutc,manv,num_acc
0,"[B01, A01]","[Cyclomoteur <50cm3, VL seul]","[Avant, None]","[3, 3]","[nan, None]","[Véhicule, None]","[nan, None]","[Traversant la chaussée, None]",201900020750
1,[A01],[VU seul 1],[nan],[3],[nan],[nan],[nan],[Même sens],201900020796
2,"[B01, A01]","[VL seul, VL seul]","[Avant gauche, Avant]",[PK ou PR ou numéro d’adresse postale croissan...,"[nan, None]","[Véhicule, Véhicule]","[nan, None]","[Sans changement de direction, Déporté A gauche]",201900020869
3,"[B01, A01]","[43, VL seul]","[Avant, None]",[PK ou PR ou numéro d’adresse postale croissan...,"[nan, None]","[Véhicule, Véhicule]","[nan, None]","[Sans changement de direction, Manœuvre de sta...",201900021309
4,"[C01, B01, A01]","[VL seul, VL seul, VL seul]","[Arrière, Arrière, Avant]",[PK ou PR ou numéro d’adresse postale décroiss...,"[nan, None, None]","[Véhicule, Véhicule, Véhicule]","[nan, None, None]","[Même sens, Même sens, Même sens]",201900018753
5,"[A01, B01]","[VL seul, VL seul]","[Avant, Côté gauche]","[3, 3]","[nan, None]","[Véhicule, Véhicule]","[nan, None]","[Tournant A gauche, Sans changement de direction]",201900037513
6,"[C01, B01, A01]","[VL seul, VL seul, VL seul]","[Côté gauche, Côté gauche, Avant]","[nan, None, None]","[nan, None, None]","[Véhicule, Véhicule, Véhicule]","[nan, None, None]","[Sans changement de direction, Sans changement...",201400048899
7,[A01],[Motocyclette > 125 cm3],[Avant],[nan],[nan],[Autre],[nan],[Entre 2 files],201400049030
8,[A01],[Autre véhicule],[Avant],[PK ou PR ou numéro d’adresse postale croissant],[nan],[Piéton],[nan],[Sans changement de direction],201700041615
9,"[B01, A01]","[Engin spécial, VL seul]","[Avant, Arrière]",[PK ou PR ou numéro d’adresse postale croissan...,"[nan, None]","[Véhicule, Véhicule]","[nan, None]","[Sans changement de direction, Arrêté (hors st...",201700042289


In [33]:
# Make a copy to avoid modifying the original
df_veh_exploded = df_vehicules.copy()

# 🚀 Explode all list-type columns together (they all have equal-length lists now)
df_veh_exploded = df_veh_exploded.explode(veh_cols, ignore_index=True)

print(f"✓ Data exploded successfully: {len(df_veh_exploded):,} rows.")

# Ajout d'un identifiant unique pour chaque véhicule
df_veh_exploded = df_veh_exploded.reset_index(drop=True)
df_veh_exploded["vehicule_id"] = df_veh_exploded.index + 1

# Réorganisation des colonnes pour mettre l'id en premier
cols = ["vehicule_id"] + [c for c in df_veh_exploded.columns if c != "vehicule_id"]
df_veh_exploded = df_veh_exploded[cols]

df_veh_exploded.head(10)

✓ Data exploded successfully: 811,335 rows.


,vehicule_id,num_veh,catv,choc,senc,obs,obsm,occutc,manv,num_acc
0,1,B01,Cyclomoteur <50cm3,Avant,3,nan,Véhicule,nan,Traversant la chaussée,201900020750
1,2,A01,VL seul,None,3,None,None,None,None,201900020750
2,3,A01,VU seul 1,nan,3,nan,nan,nan,Même sens,201900020796
3,4,B01,VL seul,Avant gauche,PK ou PR ou numéro d’adresse postale croissant,nan,Véhicule,nan,Sans changement de direction,201900020869
4,5,A01,VL seul,Avant,PK ou PR ou numéro d’adresse postale décroissant,None,Véhicule,None,Déporté A gauche,201900020869
5,6,B01,43,Avant,PK ou PR ou numéro d’adresse postale croissant,nan,Véhicule,nan,Sans changement de direction,201900021309
6,7,A01,VL seul,None,PK ou PR ou numéro d’adresse postale croissant,None,Véhicule,None,Manœuvre de stationnement,201900021309
7,8,C01,VL seul,Arrière,PK ou PR ou numéro d’adresse postale décroissant,nan,Véhicule,nan,Même sens,201900018753
8,9,B01,VL seul,Arrière,PK ou PR ou numéro d’adresse postale décroissant,None,Véhicule,None,Même sens,201900018753
9,10,A01,VL seul,Avant,PK ou PR ou numéro d’adresse postale décroissant,None,Véhicule,None,Même sens,201900018753


In [34]:
# Vérification des données après explosion
# df_veh_exploded.info()

# Each num_acc now has one row per vehicle
check = df_veh_exploded.groupby("num_acc")["num_veh"].nunique().describe()
print(check)

count    475911.000000
mean          1.704693
std           0.683776
min           1.000000
25%           1.000000
50%           2.000000
75%           2.000000
max          35.000000
Name: num_veh, dtype: float64


In [35]:
# Vérification des données après explosion

# Vérification des valeurs uniques sur les catégories
for col in ['num_veh', 'catv', 'choc', 'senc', 'obs', 'obsm', 'occutc', 'manv']:
    unique_values = df_veh_exploded[col].unique()
    print(f"\n Valeurs uniques dans '{col}': {unique_values}")

# Compter le nombre de 'nan' et '-1' dans chaque colonne catégorielle
for col in ['num_veh', 'catv', 'choc', 'senc', 'obs', 'obsm', 'occutc', 'manv']:
    nan_count = df_veh_exploded[col].isna().sum()
    neg_one_count = (df_veh_exploded[col] == '-1').sum()
    print(f"\n Colonne '{col}': NaN = {nan_count}, -1 = {neg_one_count}")


 Valeurs uniques dans 'num_veh': ['B01' 'A01' 'C01' 'B02' 'Z01' 'C03' 'D01' 'F02' 'E01' 'Y01' 'F01' 'X01'
 'D04' 'C04' 'B03' 'E05' 'C02' 'A02' 'F06' 'G07' 'H08' 'D02' 'E02' 'I01'
 'H01' 'G01' 'Z02' 'G02' 'N01' 'R01' 'H02' 'T01' 'O01' 'J01' 'K01' 'L01'
 'M01' 'I09' 'TB01' 'AA01' 'A03' 'I02' 'N03' 'L03' 'M03' 'J02' 'O03' 'K03'
 'U01' 'P01' 'BB01' 'CB01' 'V01' 'ZZ01' 'BA01' 'G06' 'H07' 'D03' 'E04'
 'F05' 'I08' 'VB01' 'A04' 'M13' 'J10' 'K11' 'L12' 'O15' 'N14' 'W01' 'A05'
 'MA01' 'C08' 'S01' 'DA01' 'Z03' 'Z04' 'VF01' 'L02' 'K02' 'P16' 'W23'
 'S19' 'Y25' 'X24' 'U21' 'Q17' 'R18' 'V22' 'T20' 'Z26' 'B28' 'A27' 'RC01'
 'G33' 'I35' 'M39' 'D30' 'N40' 'E31' 'C29' 'K37' 'Q01' 'AB01' 'PB01' 'B05'
 'X02' 'E03' '[01' 'RA01' '\\01' 'MB01' 'LB01' 'C09' 'TC01' 'BC01' 'FB01'
 'GB01' 'ZB01']

 Valeurs uniques dans 'catv': ['Cyclomoteur <50cm3' 'VL seul' 'VU seul 1' '43' 'Motocyclette > 125 cm3'
 'Autre véhicule' 'Engin spécial' 'Scooter < 50 cm3' 'Tracteur agricole'
 'Motocyclette > 50 cm3 et <= 125 cm3' '

In [36]:
# Check des dtypes des colonnes
df_veh_exploded.dtypes

vehicule_id     int64
num_veh        object
catv           object
choc           object
senc           object
obs            object
obsm           object
occutc         object
manv           object
num_acc         int64
dtype: object

In [37]:
# Check et nettoyage des colonnes catégorielles

# catv : retirer les parenthèses après "Voiturette"
# catv : 0 --> "Autre véhicule"
# catv : [50, 43, 42, 41, 60, 80] --> "Autre véhicule"
# catv : "Quad léger" & "Quad lourd" --> retirer les parenthèses
def clean_catv(value):
    if pd.isna(value):
        return value
    if value == '0':
        return 'Autre véhicule'
    if value in ['50', '43', '42', '41', '60', '80']:
        return 'Autre véhicule'
    if 'Voiturette (Quadricycle à moteur carrossé) (anciennement "voiturette ou tricycle à moteur")' in value:
        return 'Voiturette'
    if 'Quad léger <= 50 cm3 (Quadricycle à moteur non carrossé)' in value:
        return 'Quad léger <= 50 cm3'
    if 'Quad lourd > 50 cm3 (Quadricycle à moteur non carrossé)' in value:
        return 'Quad lourd > 50 cm3'
    return value

df_veh_exploded['catv'] = df_veh_exploded['catv'].apply(clean_catv)

df_veh_exploded['catv'].value_counts(dropna=False)

catv
VL seul                                502874
Motocyclette > 125 cm3                  67462
VU seul 1                               46223
Bicyclette                              37757
Scooter < 50 cm3                        30904
Cyclomoteur <50cm3                      30456
Motocyclette > 50 cm3 et <= 125 cm3     19094
Scooter > 50 cm3 et <= 125 cm3          17870
Scooter > 125 cm3                       12258
PL seul > 7                              7690
PL > 3                                   7607
Autobus                                  6070
Tracteur routier + semi-remorque         5779
Autre véhicule                           5103
Voiturette                               3871
PL seul 3                                3120
Tracteur agricole                        1685
Autocar                                  1612
Quad lourd > 50 cm3                      1120
Tramway                                  1108
Engin spécial                             891
Tracteur routier seul        

In [38]:
# clean : choc, sens, obs, obsm, occutc, manv
# common rules to apply :
# '-1' --> NaN
# 'nan' or None --> np.nan
def clean_generic(value):
    if pd.isna(value) or value in ['nan', 'None']:
        return np.nan
    if value == '-1':
        return np.nan
    return value

for col in ['choc', 'senc', 'obs', 'obsm', 'occutc', 'manv']:
    df_veh_exploded[col] = df_veh_exploded[col].apply(clean_generic)

for col in ['choc', 'senc', 'obs', 'obsm', 'occutc', 'manv']:
    print(f"\nColumn: {col}")
    print(df_veh_exploded[col].value_counts(dropna=False).head())


Column: choc
choc
Avant           291698
Avant gauche    116177
Avant droit      94729
Arrière          80746
Côté gauche      58228
Name: count, dtype: int64

Column: senc
senc
NaN                                                 464488
PK ou PR ou numéro d’adresse postale croissant      198459
PK ou PR ou numéro d’adresse postale décroissant    133352
3                                                    15036
Name: count, dtype: int64

Column: obs
obs
NaN                          705045
Véhicule en stationnement     17278
Fossé                         13196
Arbre                         11537
Glissière béton               10002
Name: count, dtype: int64

Column: obsm
obsm
Véhicule          551952
NaN               158246
Piéton             88109
Autre               9431
Animal sauvage      2209
Name: count, dtype: int64

Column: occutc
occutc
NaN    805247
1        3598
2         643
3         369
5         242
Name: count, dtype: int64

Column: manv
manv
Sans changement de direction

In [39]:

# Séparation en 5 dataframes
# 5. Usagers
df_usagers = df_source[[
    'sexe', 
    'grav', 
    'trajet', 
    'secu', 
    'secu_utl', 
    'catu', 
    'place', 
    'locp', 
    'actp', 
    'etatp', 
    'num_acc'
]].copy()

df_usagers.head()
#df_usagers.info()

,sexe,grav,trajet,secu,secu_utl,catu,place,locp,actp,etatp,num_acc
0,"Masculin,Masculin,Masculin","Blessé,Blessé,Indemne","Promenade – loisirs,Promenade – loisirs",NaN,NaN,"Conducteur,Passager,Conducteur","1,2,1",-1,"-1,Se déplaçant,Se déplaçant","-1,-1,-1",201900020750
1,Masculin,Blessé,NaN,NaN,NaN,Conducteur,1,-1,-1,-1,201900020796
2,"Masculin,Masculin,Masculin,Masculin,Masculin","Indemne,Indemne,Blessé,Indemne,Indemne","Promenade – loisirs,Promenade – loisirs,Promen...",NaN,NaN,"Conducteur,Passager,Conducteur,Passager,Passager","1,9,1,7,2",NaN,"Se déplaçant,Se déplaçant,Se déplaçant,Se dépl...","-1,-1,-1,-1,-1",201900020869
3,Masculin,Blessé,NaN,NaN,NaN,Conducteur,1,-1,-1,-1,201900021309
4,"Féminin,Masculin,Masculin,Féminin,Masculin,Fém...","Indemne,Blessé,Indemne,Indemne,Indemne,Indemne","Promenade – loisirs,Promenade – loisirs,Promen...",NaN,NaN,"Conducteur,Passager,Passager,Passager,Passager...","1,2,2,3,4,1","-1,-1,-1,-1,-1,-1","Se déplaçant,Se déplaçant,Se déplaçant,Se dépl...","-1,-1,-1,-1,-1,-1",201900018753


# ZOUBIR

In [44]:
print(df_accidents.head())

        num_acc                         lum                 agg  int      atm  \
0  201900020750                  Plein jour    En agglomération    2  Normale   
1  201900020796  Nuit sans éclairage public  Hors agglomération    1  Normale   
2  201900020869                  Plein jour    En agglomération    1  Normale   
3  201900021309                  Plein jour    En agglomération    1  Normale   
4  201900018753  Nuit sans éclairage public  Hors agglomération    1  Normale   

                       adr                             col              circ  \
0            NAGIS(RUE DE)    Deux véhicules – par le coté  Bidirectionnelle   
1                   RD 569                  Sans collision               NaN   
2       Avenue Victor Hugo       Deux véhicules - frontale  Bidirectionnelle   
3  MAURICE BERTHEAUX ALLEE  Deux véhicules – par l’arrière     A sens unique   
4                      A86                 Autre collision     A sens unique   

                plan  prof      

In [45]:
print(df_lieux.head())

   com_code               com_name  dep_code           dep_name  reg_code  \
0   91174.0       Corbeil-Essonnes      91.0            Essonne      11.0   
1   13047.0                 Istres      13.0   Bouches-du-Rhône      93.0   
2   38412.0  Saint-Laurent-du-Pont      38.0              Isère      84.0   
3   93046.0           Livry-Gargan      93.0  Seine-Saint-Denis      11.0   
4   92036.0          Gennevilliers      92.0     Hauts-de-Seine      11.0   

                     reg_name                  catr  nbv vosp       num_acc  
0               Île-de-France                 autre    4  NaN  201900020750  
1  Provence-Alpes-Côte d'Azur  Route Départementale    2  NaN  201900020796  
2        Auvergne-Rhône-Alpes  Route Départementale    2  NaN  201900020869  
3               Île-de-France        Voie Communale    1  NaN  201900021309  
4               Île-de-France             Autoroute    2  NaN  201900018753  


In [ ]:
print(df_vehicules.head())

In [55]:
# Import du mapping depuis le fichier mapping_data.py
from mapping_data import MAPPING_DATA

# Fonction pour créer un DataFrame de correspondance
def create_mapping_df(mapping_dict, table_name):
    """
    Crée un DataFrame de correspondance à partir d'un dictionnaire de mapping
    
    Args:
        mapping_dict (dict): Dictionnaire avec clé -> libellé
        table_name (str): Nom de la table de correspondance
    
    Returns:
        pd.DataFrame: DataFrame avec colonnes 'code' et 'libelle'
    """
    df = pd.DataFrame([
        {'code': code, 'libelle': libelle} 
        for code, libelle in mapping_dict.items()
    ])
    df['table_name'] = table_name
    return df

# Créer tous les DataFrames de correspondance
correspondance_dfs = {}

for column_name, mapping_dict in MAPPING_DATA.items():
    table_name = f"ref_{column_name}"
    correspondance_dfs[table_name] = create_mapping_df(mapping_dict, table_name)
    print(f"Table {table_name} créée avec {len(correspondance_dfs[table_name])} entrées")

# Afficher un exemple
print("\nExemple - Table ref_lum:")
print(correspondance_dfs['ref_agg'])

Table ref_lum créée avec 5 entrées
Table ref_agg créée avec 2 entrées
Table ref_int créée avec 9 entrées
Table ref_atm créée avec 9 entrées
Table ref_col créée avec 7 entrées
Table ref_gps créée avec 5 entrées
Table ref_catr créée avec 7 entrées
Table ref_circ créée avec 4 entrées
Table ref_vosp créée avec 3 entrées
Table ref_prof créée avec 4 entrées
Table ref_plan créée avec 4 entrées
Table ref_surf créée avec 9 entrées
Table ref_infra créée avec 7 entrées
Table ref_situ créée avec 5 entrées
Table ref_senc créée avec 2 entrées
Table ref_catv créée avec 33 entrées
Table ref_obs créée avec 16 entrées
Table ref_obsm créée avec 6 entrées
Table ref_choc créée avec 9 entrées
Table ref_manv créée avec 24 entrées
Table ref_catu créée avec 4 entrées
Table ref_grav créée avec 4 entrées
Table ref_sexe créée avec 2 entrées
Table ref_trajet créée avec 6 entrées
Table ref_secu créée avec 5 entrées
Table ref_secu_utl créée avec 3 entrées
Table ref_lopc créée avec 8 entrées
Table ref_actp créée avec

In [56]:
# Créer une table de correspondance consolidée (optionnel)
df_correspondances_all = pd.concat(correspondance_dfs.values(), ignore_index=True)
df_correspondances_all = df_correspondances_all[['table_name', 'code', 'libelle']]

print(f"Table de correspondances consolidée: {len(df_correspondances_all)} entrées")
print("\nPremières lignes:")
print(df_correspondances_all.head(10))

print("\nNombre d'entrées par table:")
print(df_correspondances_all['table_name'].value_counts())

Table de correspondances consolidée: 213 entrées

Premières lignes:
  table_name code                                libelle
0    ref_lum    1                             Plein jour
1    ref_lum    2                     Crépuscule ou aube
2    ref_lum    3             Nuit sans éclairage public
3    ref_lum    4  Nuit avec éclairage public non allumé
4    ref_lum    5      Nuit avec éclairage public allumé
5    ref_agg    1                     Hors agglomération
6    ref_agg    2                       En agglomération
7    ref_int    1                      Hors intersection
8    ref_int    2                      Intersection en X
9    ref_int    3                      Intersection en T

Nombre d'entrées par table:
table_name
ref_catv        33
ref_manv        24
ref_obs         16
ref_atm          9
ref_int          9
ref_choc         9
ref_surf         9
ref_actp         8
ref_lopc         8
ref_infra        7
ref_col          7
ref_catr         7
ref_obsm         6
ref_trajet       6

## Mapping des catégories dans les DataFrames

Maintenant nous allons mapper chaque catégorie par sa clé dans nos DataFrames `df_accidents`, `df_lieux`, et `df_vehicules`.

In [48]:
def map_categorical_columns(df, df_name, column_mappings):
    """
    Mappe les colonnes catégorielles d'un DataFrame selon les dictionnaires de correspondance
    
    Args:
        df (pd.DataFrame): DataFrame à mapper
        df_name (str): Nom du DataFrame pour les logs
        column_mappings (dict): Dictionnaire {colonne: mapping_dict}
    
    Returns:
        pd.DataFrame: DataFrame avec colonnes mappées
    """
    df_mapped = df.copy()
    
    print(f"\n=== Mapping {df_name} ===")
    
    for column, mapping_dict in column_mappings.items():
        if column in df_mapped.columns:
            # Vérifier les valeurs avant mapping
            unique_before = df_mapped[column].unique()
            missing_keys = [val for val in unique_before if pd.notna(val) and val not in mapping_dict]
            
            if missing_keys:
                print(f"⚠️  {column}: Valeurs non trouvées dans le mapping: {missing_keys}")
            
            # Appliquer le mapping
            df_mapped[f"{column}_libelle"] = df_mapped[column].map(mapping_dict)
            
            # Statistiques
            mapped_count = df_mapped[f"{column}_libelle"].notna().sum()
            total_count = len(df_mapped)
            print(f"✓ {column}: {mapped_count}/{total_count} valeurs mappées")
        else:
            print(f"❌ {column}: Colonne non trouvée dans {df_name}")
    
    return df_mapped

# Définir les mappings pour chaque DataFrame
accidents_mappings = {
    'lum': MAPPING_DATA['lum'],
    'agg': MAPPING_DATA['agg'],
    'int': MAPPING_DATA['int'],
    'atm': MAPPING_DATA['atm'],
    'col': MAPPING_DATA['col'],
    'circ': MAPPING_DATA['circ'],
    'plan': MAPPING_DATA['plan'],
    'prof': MAPPING_DATA['prof'],
    'surf': MAPPING_DATA['surf']
}

lieux_mappings = {
    'catr': MAPPING_DATA['catr'],
    'vosp': MAPPING_DATA['vosp']
}

vehicules_mappings = {
    'catv': MAPPING_DATA['catv'],
    'choc': MAPPING_DATA['choc'],
    'senc': MAPPING_DATA['senc'],
    'obs': MAPPING_DATA['obs'],
    'obsm': MAPPING_DATA['obsm'],
    'manv': MAPPING_DATA['manv']
}

In [49]:
# Appliquer le mapping sur df_accidents
print("Avant mapping - df_accidents:")
print(f"Shape: {df_accidents.shape}")
print(f"Colonnes: {list(df_accidents.columns)}")

df_accidents_mapped = map_categorical_columns(df_accidents, "df_accidents", accidents_mappings)

print(f"\nAprès mapping - df_accidents_mapped:")
print(f"Shape: {df_accidents_mapped.shape}")
print(f"Nouvelles colonnes ajoutées: {[col for col in df_accidents_mapped.columns if col.endswith('_libelle')]}")

Avant mapping - df_accidents:
Shape: (475911, 14)
Colonnes: ['num_acc', 'lum', 'agg', 'int', 'atm', 'adr', 'col', 'circ', 'plan', 'prof', 'surf', 'infra', 'situ', 'year_georef']

=== Mapping df_accidents ===
⚠️  lum: Valeurs non trouvées dans le mapping: ['Plein jour', 'Nuit sans éclairage public', 'Nuit avec éclairage public non allumé', 'Nuit avec éclairage public allumé', 'Crépuscule ou aube']
✓ lum: 0/475911 valeurs mappées
⚠️  agg: Valeurs non trouvées dans le mapping: ['En agglomération', 'Hors agglomération']
✓ agg: 0/475911 valeurs mappées
⚠️  int: Valeurs non trouvées dans le mapping: [np.int64(0)]
✓ int: 475821/475911 valeurs mappées
⚠️  atm: Valeurs non trouvées dans le mapping: ['Normale', 'Temps couvert', 'Temps éblouissant', 'Pluie légère', 'Pluie forte', 'Autre', 'Vent fort - tempête', 'Brouillard - fumée', 'Neige - grêle']
✓ atm: 0/475911 valeurs mappées
⚠️  col: Valeurs non trouvées dans le mapping: ['Deux véhicules – par le coté', 'Sans collision', 'Deux véhicules - f

In [57]:
# Appliquer le mapping sur df_lieux
print("Avant mapping - df_lieux:")
print(f"Shape: {df_lieux.shape}")
print(f"Colonnes: {list(df_lieux.columns)}")

df_lieux_mapped = map_categorical_columns(df_lieux, "df_lieux", lieux_mappings)

print(f"\nAprès mapping - df_lieux_mapped:")
print(f"Shape: {df_lieux_mapped.shape}")
print(f"Nouvelles colonnes ajoutées: {[col for col in df_lieux_mapped.columns if col.endswith('_libelle')]}")

Avant mapping - df_lieux:
Shape: (475911, 10)
Colonnes: ['com_code', 'com_name', 'dep_code', 'dep_name', 'reg_code', 'reg_name', 'catr', 'nbv', 'vosp', 'num_acc']

=== Mapping df_lieux ===
⚠️  catr: Valeurs non trouvées dans le mapping: ['autre', 'Route Départementale', 'Voie Communale', 'Autoroute', 'Route Nationale', 'Parc de stationnement ouvert à la circulation publique', 'Hors réseau public']
✓ catr: 0/475911 valeurs mappées
⚠️  vosp: Valeurs non trouvées dans le mapping: ['Voie réservée', 'Piste cyclable', 'Banque cyclable']
✓ vosp: 0/475911 valeurs mappées

Après mapping - df_lieux_mapped:
Shape: (475911, 12)
Nouvelles colonnes ajoutées: ['catr_libelle', 'vosp_libelle']


In [ ]:
# Appliquer le mapping sur df_veh_exploded (DataFrame des véhicules)
print("Avant mapping - df_veh_exploded:")
print(f"Shape: {df_veh_exploded.shape}")
print(f"Colonnes: {list(df_veh_exploded.columns)}")

df_vehicules_mapped = map_categorical_columns(df_veh_exploded, "df_veh_exploded", vehicules_mappings)

print(f"\nAprès mapping - df_vehicules_mapped:")
print(f"Shape: {df_vehicules_mapped.shape}")
print(f"Nouvelles colonnes ajoutées: {[col for col in df_vehicules_mapped.columns if col.endswith('_libelle')]}")

In [58]:
# Vérification des données mappées - Exemples
print("=== EXEMPLES DE DONNÉES MAPPÉES ===")

print("\n1. df_accidents_mapped - Luminosité:")
example_lum = df_accidents_mapped[['lum', 'lum_libelle']].dropna().head()
print(example_lum)

print("\n2. df_lieux_mapped - Catégorie de route:")
example_catr = df_lieux_mapped[['catr', 'catr_libelle']].dropna().head()
print(example_catr)

print("\n3. df_vehicules_mapped - Catégorie de véhicule:")
example_catv = df_vehicules_mapped[['catv', 'catv_libelle']].dropna().head()
print(example_catv)

# Statistiques générales
print("\n=== STATISTIQUES DE MAPPING ===")
print(f"df_accidents_mapped: {len([col for col in df_accidents_mapped.columns if col.endswith('_libelle')])} colonnes mappées")
print(f"df_lieux_mapped: {len([col for col in df_lieux_mapped.columns if col.endswith('_libelle')])} colonnes mappées")
print(f"df_vehicules_mapped: {len([col for col in df_vehicules_mapped.columns if col.endswith('_libelle')])} colonnes mappées")

=== EXEMPLES DE DONNÉES MAPPÉES ===

1. df_accidents_mapped - Luminosité:
Empty DataFrame
Columns: [lum, lum_libelle]
Index: []

2. df_lieux_mapped - Catégorie de route:
Empty DataFrame
Columns: [catr, catr_libelle]
Index: []

3. df_vehicules_mapped - Catégorie de véhicule:


NameError: name 'df_vehicules_mapped' is not defined

In [ ]:
# Contrôle qualité du mapping
print("=== CONTRÔLE QUALITÉ DU MAPPING ===")

def check_mapping_quality(df, df_name):
    """
    Vérifie la qualité du mapping d'un DataFrame
    """
    print(f"\n--- {df_name} ---")
    
    libelle_cols = [col for col in df.columns if col.endswith('_libelle')]
    
    for col in libelle_cols:
        original_col = col.replace('_libelle', '')
        
        # Compter les valeurs non nulles dans l'original
        original_not_null = df[original_col].notna().sum()
        
        # Compter les valeurs mappées avec succès
        mapped_success = df[col].notna().sum()
        
        # Pourcentage de réussite
        success_rate = (mapped_success / original_not_null * 100) if original_not_null > 0 else 0
        
        print(f"{original_col}: {mapped_success}/{original_not_null} mappées ({success_rate:.1f}%)")
        
        # Identifier les valeurs non mappées
        unmapped = df[df[original_col].notna() & df[col].isna()][original_col].unique()
        if len(unmapped) > 0:
            print(f"  ❌ Valeurs non mappées: {unmapped}")

# Vérifier la qualité pour chaque DataFrame
check_mapping_quality(df_accidents_mapped, "Accidents")
check_mapping_quality(df_lieux_mapped, "Lieux")
check_mapping_quality(df_vehicules_mapped, "Véhicules")

In [50]:
# Problème détecté : certaines colonnes contiennent déjà les libellés au lieu des codes
# Créons des mappings inverses pour récupérer les codes

def create_reverse_mapping(original_mapping):
    """Créer un mapping inverse libellé -> code"""
    return {v: k for k, v in original_mapping.items()}

# Créer les mappings inverses pour les colonnes qui en ont besoin
reverse_mappings = {}
for col_name, mapping_dict in MAPPING_DATA.items():
    reverse_mappings[col_name] = create_reverse_mapping(mapping_dict)

# Fonction pour normaliser les libellés (gérer les variations)
def normalize_label(label):
    """Normalise les libellés pour le matching"""
    if pd.isna(label):
        return label
    
    # Conversions spécifiques observées dans les données
    normalizations = {
        # LUM
        'Plein jour': 'Plein jour',
        'Crépuscule ou aube': 'Crépuscule ou aube', 
        'Nuit sans éclairage public': 'Nuit sans éclairage public',
        'Nuit avec éclairage public non allumé': 'Nuit avec éclairage public non allumé',
        'Nuit avec éclairage public allumé': 'Nuit avec éclairage public allumé',
        
        # AGG
        'En agglomération': 'En agglomération',
        'Hors agglomération': 'Hors agglomération',
        
        # ATM
        'Normale': 'Normale',
        'Temps couvert': 'Temps couvert',
        'Temps éblouissant': 'Temps éblouissant',
        'Pluie légère': 'Pluie légère',
        'Pluie forte': 'Pluie forte',
        'Autre': 'Autre',
        'Vent fort - tempête': 'Vent fort ou tempête',
        'Brouillard - fumée': 'Brouillard ou fumée',
        'Neige - grêle': 'Neige ou grêle',
        
        # COL
        'Deux véhicules – par le coté': 'Deux véhicules - par le côté',
        'Sans collision': 'Sans collision',
        'Deux véhicules - frontale': 'Deux véhicules - frontale',
        'Deux véhicules – par l\'arrière': 'Deux véhicules - par l\'arrière',
        'Autre collision': 'Autre collision',
        'Trois véhicules et plus - collisions multiples': 'Trois véhicules et plus - collisions multiples',
        'Trois véhicules et plus – en chaîne': 'Trois véhicules et plus - en chaîne',
        
        # CIRC
        'Bidirectionnelle': 'Bidirectionnelle',
        'A sens unique': 'À sens unique',
        'A chaussées séparées': 'À chaussées séparées',
        'Avec voies d\'affectation variable': 'Avec voies d\'affectation variable',
        
        # PLAN
        'Partie rectiligne': 'Partie rectiligne',
        'En courbe à droite': 'En courbe à droite',
        'En courbe à gauche': 'En courbe à gauche',
        'En « S »': 'En S',
        
        # PROF
        'Plat': 'Plat',
        'Pente': 'Pente',
        'Sommet de côte': 'Sommet de côte',
        'Bas de côte': 'Bas de côte',
        
        # SURF
        'mouillée': 'Mouillée',
        'normale': 'Normale',
        'autre': 'Autre',
        'enneigée': 'Enneigée',
        'flaques': 'Flaques',
        'corps gras - huile': 'Corps gras ou huile',
        'verglacée': 'Verglacée',
        'inondée': 'Inondée',
        'boue': 'Boue'
    }
    
    return normalizations.get(str(label), str(label))

print("Mappings inverses créés pour récupérer les codes à partir des libellés")

Mappings inverses créés pour récupérer les codes à partir des libellés


In [51]:
def map_labels_to_codes(df, df_name, column_mappings):
    """
    Mappe les libellés vers les codes numériques
    
    Args:
        df (pd.DataFrame): DataFrame à mapper
        df_name (str): Nom du DataFrame pour les logs
        column_mappings (dict): Dictionnaire {colonne: mapping_dict}
    
    Returns:
        pd.DataFrame: DataFrame avec colonnes de codes ajoutées
    """
    df_mapped = df.copy()
    
    print(f"\n=== Mapping des libellés vers codes - {df_name} ===")
    
    for column, mapping_dict in column_mappings.items():
        if column in df_mapped.columns:
            # Créer le mapping inverse
            reverse_mapping = {v: k for k, v in mapping_dict.items()}
            
            # Normaliser les valeurs avant mapping
            normalized_values = df_mapped[column].apply(normalize_label)
            
            # Appliquer le mapping inverse pour obtenir les codes
            df_mapped[f"{column}_code"] = normalized_values.map(reverse_mapping)
            
            # Garder les libellés originaux dans une colonne séparée
            df_mapped[f"{column}_libelle"] = df_mapped[column]
            
            # Remplacer la colonne originale par les codes
            df_mapped[column] = df_mapped[f"{column}_code"]
            
            # Statistiques
            mapped_count = df_mapped[f"{column}_code"].notna().sum()
            total_count = len(df_mapped)
            unmapped = df_mapped[df_mapped[column].isna() & normalized_values.notna()]
            
            print(f"✓ {column}: {mapped_count}/{total_count} valeurs mappées")
            
            if len(unmapped) > 0:
                unique_unmapped = unmapped[f"{column}_libelle"].unique()[:5]  # Limiter l'affichage
                print(f"  ⚠️ Exemples de valeurs non mappées: {unique_unmapped}")
        else:
            print(f"❌ {column}: Colonne non trouvée dans {df_name}")
    
    return df_mapped

# Appliquer le nouveau mapping sur df_accidents
print("Application du mapping libellés -> codes sur df_accidents")
df_accidents_final = map_labels_to_codes(df_accidents, "df_accidents", accidents_mappings)

Application du mapping libellés -> codes sur df_accidents

=== Mapping des libellés vers codes - df_accidents ===
✓ lum: 475911/475911 valeurs mappées
✓ agg: 475911/475911 valeurs mappées
✓ int: 0/475911 valeurs mappées
  ⚠️ Exemples de valeurs non mappées: [2 1 9 6 3]
✓ atm: 475860/475911 valeurs mappées
✓ col: 417547/475911 valeurs mappées
  ⚠️ Exemples de valeurs non mappées: ['Deux véhicules – par l’arrière']
✓ circ: 447350/475911 valeurs mappées
✓ plan: 441476/475911 valeurs mappées
✓ prof: 447000/475911 valeurs mappées
✓ surf: 459759/475911 valeurs mappées


In [52]:
# Traitement spécial pour la colonne 'int' qui contient déjà des codes numériques
print("\n=== Traitement spécial pour la colonne 'int' ===")

# Vérifier les valeurs uniques dans 'int'
print(f"Valeurs uniques dans 'int': {sorted(df_accidents['int'].unique())}")

# La colonne 'int' contient déjà des codes (0, 1, 2, etc.)
# Mappage direct vers les libellés
df_accidents_final['int_libelle'] = df_accidents_final['int'].map(MAPPING_DATA['int'])
df_accidents_final['int_code'] = df_accidents_final['int']

# Statistiques pour 'int'
mapped_int = df_accidents_final['int_libelle'].notna().sum()
total_int = df_accidents_final['int'].notna().sum()
print(f"Colonne 'int': {mapped_int}/{total_int} valeurs mappées")

# Vérifier les valeurs non mappées
unmapped_int = df_accidents_final[df_accidents_final['int'].notna() & df_accidents_final['int_libelle'].isna()]['int'].unique()
if len(unmapped_int) > 0:
    print(f"Valeurs 'int' non mappées: {unmapped_int}")

print(f"\nShape finale df_accidents_final: {df_accidents_final.shape}")
print(f"Colonnes ajoutées: {[col for col in df_accidents_final.columns if col.endswith(('_code', '_libelle'))]}")

# Exemple du résultat final pour df_accidents
print("\n=== EXEMPLE DE RÉSULTAT FINAL - df_accidents ===")

example_cols = ['num_acc', 'lum', 'lum_libelle', 'agg', 'agg_libelle', 'int', 'int_libelle']
print(df_accidents_final[example_cols].head())

print("\n=== RÉSUMÉ DES COLONNES MAPPÉES ===")
for col in ['lum', 'agg', 'int', 'atm', 'col', 'circ', 'plan', 'prof', 'surf']:
    if f'{col}_code' in df_accidents_final.columns or f'{col}_libelle' in df_accidents_final.columns:
        code_count = df_accidents_final[col].notna().sum() if col in df_accidents_final.columns else 0
        libelle_count = df_accidents_final[f'{col}_libelle'].notna().sum() if f'{col}_libelle' in df_accidents_final.columns else 0
        total = len(df_accidents_final)
        print(f"{col}: {code_count} codes, {libelle_count} libellés sur {total} total")


=== Traitement spécial pour la colonne 'int' ===
Valeurs uniques dans 'int': [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
Colonne 'int': 0/0 valeurs mappées

Shape finale df_accidents_final: (475911, 32)
Colonnes ajoutées: ['lum_code', 'lum_libelle', 'agg_code', 'agg_libelle', 'int_code', 'int_libelle', 'atm_code', 'atm_libelle', 'col_code', 'col_libelle', 'circ_code', 'circ_libelle', 'plan_code', 'plan_libelle', 'prof_code', 'prof_libelle', 'surf_code', 'surf_libelle']

=== EXEMPLE DE RÉSULTAT FINAL - df_accidents ===
        num_acc  lum                 lum_libelle  agg         agg_libelle  \
0  201900020750    1                  Plein jour    2    En agglomération   
1  201900020796    3  Nuit sans éclairage public    1  Hors agglomération   
2  201900020869    1                  Plein jour    2    En agglomération   
3  201900021309    1                  Plein jour    2    En agglomération   
4  

In [ ]:
# Correction pour la colonne 'int'
print("\n=== CORRECTION pour la colonne 'int' ===")

# La colonne 'int' dans le df original contient les codes
original_int_values = df_accidents['int'].unique()
print(f"Valeurs originales dans 'int': {sorted([x for x in original_int_values if pd.notna(x)])}")

# Restaurer les valeurs originales de 'int' et mapper correctement
df_accidents_final['int'] = df_accidents['int']  # Restaurer les codes originaux
df_accidents_final['int_code'] = df_accidents['int']  # Les codes
df_accidents_final['int_libelle'] = df_accidents['int'].map(MAPPING_DATA['int'])  # Les libellés

# Statistiques pour 'int' corrigées
mapped_int = df_accidents_final['int_libelle'].notna().sum()
total_int = df_accidents_final['int'].notna().sum()
print(f"Colonne 'int' corrigée: {mapped_int}/{total_int} valeurs mappées")

# Vérifier les valeurs non mappées
unmapped_int = df_accidents_final[df_accidents_final['int'].notna() & df_accidents_final['int_libelle'].isna()]['int'].unique()
if len(unmapped_int) > 0:
    print(f"Valeurs 'int' non mappées: {unmapped_int}")
    # Le code 0 n'est pas dans notre mapping, ajoutons-le
    if 0 in unmapped_int:
        print("Le code 0 pour 'int' n'est pas dans le mapping. Valeur probablement pour 'non renseigné' ou 'autre'.")

# Afficher un exemple avec 'int' corrigé
print("\n=== EXEMPLE AVEC 'int' CORRIGÉ ===")
example_with_int = df_accidents_final[df_accidents_final['int'].notna()][['num_acc', 'int', 'int_libelle']].head()
print(example_with_int)

## Traitement des DataFrames lieux et véhicules

Maintenant appliquons le même traitement sur `df_lieux` et `df_veh_exploded`.

In [ ]:
# Traitement de df_lieux - certaines colonnes sont en libellés, d'autres en codes
print("=== TRAITEMENT df_lieux ===")

# Vérifier les types de données dans df_lieux
print("\nTypes de données dans df_lieux:")
for col in ['catr', 'vosp']:
    if col in df_lieux.columns:
        unique_vals = df_lieux[col].unique()[:10]  # Limiter l'affichage
        print(f"{col}: {unique_vals}")

# La colonne 'catr' contient des libellés, 'vosp' semble contenir des codes
# Traitement spécifique pour chaque colonne

df_lieux_final = df_lieux.copy()

# Pour CATR (catégorie de route) - contient des libellés
if 'catr' in df_lieux_final.columns:
    # Créer le mapping inverse pour catr
    catr_reverse = {v: k for k, v in MAPPING_DATA['catr'].items()}
    
    # Normalisation spécifique pour catr
    def normalize_catr(value):
        if pd.isna(value):
            return value
        normalizations = {
            'Autoroute': 'Autoroute',
            'Route nationale': 'Route nationale', 
            'Route Départementale': 'Route départementale',
            'Voie Communale': 'Voie communale',
            'autre': 'Autre',
            'Autre': 'Autre'
        }
        return normalizations.get(str(value), str(value))
    
    # Appliquer le mapping pour catr
    normalized_catr = df_lieux_final['catr'].apply(normalize_catr)
    df_lieux_final['catr_code'] = normalized_catr.map(catr_reverse)
    df_lieux_final['catr_libelle'] = df_lieux_final['catr']
    df_lieux_final['catr'] = df_lieux_final['catr_code']
    
    # Statistiques catr
    mapped_catr = df_lieux_final['catr_code'].notna().sum()
    total_catr = len(df_lieux_final)
    print(f"\nCATR mappé: {mapped_catr}/{total_catr}")
    
    # Valeurs non mappées
    unmapped_catr = df_lieux_final[df_lieux_final['catr'].isna() & df_lieux_final['catr_libelle'].notna()]['catr_libelle'].unique()
    if len(unmapped_catr) > 0:
        print(f"Valeurs CATR non mappées: {unmapped_catr}")

# Pour VOSP (voie spéciale) - vérifier si codes ou libellés
if 'vosp' in df_lieux_final.columns:
    # Vérifier le type de données
    vosp_sample = df_lieux_final['vosp'].dropna().head(10)
    print(f"\nÉchantillon VOSP: {vosp_sample.values}")
    
    # Si ce sont des codes numériques, mapper directement
    try:
        # Tenter de convertir en numérique
        vosp_numeric = pd.to_numeric(df_lieux_final['vosp'], errors='coerce')
        if vosp_numeric.notna().sum() > 0:
            # Ce sont des codes numériques
            df_lieux_final['vosp_code'] = vosp_numeric
            df_lieux_final['vosp_libelle'] = vosp_numeric.map(MAPPING_DATA['vosp'])
            df_lieux_final['vosp'] = df_lieux_final['vosp_code']
            
            mapped_vosp = df_lieux_final['vosp_libelle'].notna().sum()
            total_vosp = df_lieux_final['vosp_code'].notna().sum()
            print(f"VOSP mappé: {mapped_vosp}/{total_vosp}")
        else:
            # Ce sont des libellés
            vosp_reverse = {v: k for k, v in MAPPING_DATA['vosp'].items()}
            df_lieux_final['vosp_code'] = df_lieux_final['vosp'].map(vosp_reverse)
            df_lieux_final['vosp_libelle'] = df_lieux_final['vosp']
            df_lieux_final['vosp'] = df_lieux_final['vosp_code']
    except:
        print("Erreur lors du traitement de VOSP")

print(f"\nShape finale df_lieux_final: {df_lieux_final.shape}")

In [ ]:
# Traitement de df_veh_exploded - les colonnes contiennent principalement des libellés
print("\n=== TRAITEMENT df_veh_exploded ===")

# Vérifier les types de données dans df_veh_exploded pour les colonnes catégorielles
print("\nTypes de données dans df_veh_exploded:")
veh_categorical_cols = ['catv', 'choc', 'senc', 'obs', 'obsm', 'manv']
for col in veh_categorical_cols:
    if col in df_veh_exploded.columns:
        unique_vals = df_veh_exploded[col].unique()[:5]  # Limiter l'affichage
        print(f"{col}: {unique_vals}")

# Appliquer le mapping en vérifiant chaque colonne
df_vehicules_final = df_veh_exploded.copy()

for col in veh_categorical_cols:
    if col in df_vehicules_final.columns:
        print(f"\n--- Traitement de {col} ---")
        
        # Vérifier si la colonne contient des codes numériques ou des libellés
        sample_values = df_vehicules_final[col].dropna().head(10)
        
        # Essayer de détecter si ce sont des codes numériques
        try:
            numeric_values = pd.to_numeric(df_vehicules_final[col], errors='coerce')
            numeric_count = numeric_values.notna().sum()
            total_count = df_vehicules_final[col].notna().sum()
            
            if numeric_count > total_count * 0.8:  # Si >80% sont numériques, traiter comme codes
                print(f"{col}: Détecté comme codes numériques")
                df_vehicules_final[f'{col}_code'] = numeric_values
                df_vehicules_final[f'{col}_libelle'] = numeric_values.map(MAPPING_DATA[col])
                df_vehicules_final[col] = df_vehicules_final[f'{col}_code']
            else:
                print(f"{col}: Détecté comme libellés")
                # Traiter comme libellés
                reverse_mapping = {v: k for k, v in MAPPING_DATA[col].items()}
                
                # Normalisation spécifique pour les véhicules
                def normalize_vehicle_label(value):
                    if pd.isna(value) or value in ['nan', 'None', None]:
                        return np.nan
                    return str(value)
                
                normalized_values = df_vehicules_final[col].apply(normalize_vehicle_label)
                df_vehicules_final[f'{col}_code'] = normalized_values.map(reverse_mapping)
                df_vehicules_final[f'{col}_libelle'] = df_vehicules_final[col]
                df_vehicules_final[col] = df_vehicules_final[f'{col}_code']
                
        except Exception as e:
            print(f"Erreur lors du traitement de {col}: {e}")
            continue
            
        # Statistiques
        if f'{col}_code' in df_vehicules_final.columns:
            mapped_count = df_vehicules_final[f'{col}_code'].notna().sum()
            total_count = len(df_vehicules_final)
            print(f"{col}: {mapped_count}/{total_count} valeurs mappées")
            
            # Valeurs non mappées
            unmapped = df_vehicules_final[
                df_vehicules_final[f'{col}_libelle'].notna() & 
                df_vehicules_final[f'{col}_code'].isna()
            ][f'{col}_libelle'].unique()[:5]
            
            if len(unmapped) > 0:
                print(f"  Exemples non mappés: {unmapped}")

print(f"\nShape finale df_vehicules_final: {df_vehicules_final.shape}")

In [ ]:
# Résumé final et exemples
print("\n" + "="*80)
print("RÉSUMÉ FINAL DU MAPPING")
print("="*80)

# Statistiques pour df_accidents_final
print("\n1. ACCIDENTS:")
print(f"   Shape: {df_accidents_final.shape}")
accident_mapped_cols = [col for col in df_accidents_final.columns if col.endswith('_libelle')]
print(f"   Colonnes mappées: {len(accident_mapped_cols)}")
print(f"   Colonnes: {accident_mapped_cols}")

# Statistiques pour df_lieux_final
print("\n2. LIEUX:")
print(f"   Shape: {df_lieux_final.shape}")
lieux_mapped_cols = [col for col in df_lieux_final.columns if col.endswith('_libelle')]
print(f"   Colonnes mappées: {len(lieux_mapped_cols)}")
print(f"   Colonnes: {lieux_mapped_cols}")

# Statistiques pour df_vehicules_final
print("\n3. VÉHICULES:")
print(f"   Shape: {df_vehicules_final.shape}")
vehicules_mapped_cols = [col for col in df_vehicules_final.columns if col.endswith('_libelle')]
print(f"   Colonnes mappées: {len(vehicules_mapped_cols)}")
print(f"   Colonnes: {vehicules_mapped_cols}")

# Exemples de données finales
print("\n" + "="*50)
print("EXEMPLES DE DONNÉES MAPPÉES")
print("="*50)

print("\n1. Exemple df_accidents_final:")
example_accident = df_accidents_final[['num_acc', 'lum', 'lum_libelle', 'agg', 'agg_libelle']].head(3)
print(example_accident)

print("\n2. Exemple df_lieux_final:")
example_lieux = df_lieux_final[['num_acc', 'catr', 'catr_libelle'] + 
                               ([('vosp', 'vosp_libelle')] if 'vosp_libelle' in df_lieux_final.columns else [])].head(3)
print(example_lieux)

print("\n3. Exemple df_vehicules_final:")
example_vehicules_cols = ['vehicule_id', 'num_acc', 'catv', 'catv_libelle'] if 'catv_libelle' in df_vehicules_final.columns else ['vehicule_id', 'num_acc', 'catv']
example_vehicules = df_vehicules_final[example_vehicules_cols].head(3)
print(example_vehicules)

In [ ]:
# Préparer les DataFrames finaux pour l'export vers PostgreSQL
print("\n" + "="*80)
print("PRÉPARATION POUR EXPORT POSTGRESQL")
print("="*80)

# Créer un dictionnaire avec tous les DataFrames prêts pour l'export
data_for_export = {
    # Tables principales
    'accidents': df_accidents_final,
    'lieux': df_lieux_final, 
    'vehicules': df_vehicules_final,
    'date_accident': df_date_accident,  # DataFrame créé précédemment
    
    # Tables de correspondance
    'correspondances_all': df_correspondances_all
}

# Ajouter les tables de correspondance individuelles
for table_name, df_ref in correspondance_dfs.items():
    data_for_export[table_name] = df_ref

print(f"\nDataFrames prêts pour export: {len(data_for_export)}")
print("Tables principales:")
for name in ['accidents', 'lieux', 'vehicules', 'date_accident']:
    if name in data_for_export:
        print(f"  - {name}: {data_for_export[name].shape}")

print("\nTables de référence:")
for name in data_for_export.keys():
    if name.startswith('ref_'):
        print(f"  - {name}: {data_for_export[name].shape}")

print("\n✅ Mapping terminé avec succès !")
print("\nProchaines étapes:")
print("1. Exporter les DataFrames vers PostgreSQL")
print("2. Créer les contraintes de clés étrangères")
print("3. Valider l'intégrité des données")
print("4. Créer les index pour optimiser les performances")

In [ ]:
# Fonction utilitaire pour exporter vers PostgreSQL
def export_dataframe_to_postgres(df, table_name, db_config, db_name, if_exists='replace'):
    """
    Exporte un DataFrame vers PostgreSQL
    
    Args:
        df (pd.DataFrame): DataFrame à exporter
        table_name (str): Nom de la table dans PostgreSQL
        db_config (dict): Configuration de la base
        db_name (str): Nom de la base de données
        if_exists (str): Action si la table existe ('replace', 'append', 'fail')
    
    Returns:
        bool: True si succès, False sinon
    """
    try:
        # Construction de l'URL de connexion
        db_url = (
            f"postgresql+psycopg2://{db_config['user']}:{db_config['password']}"
            f"@{db_config['host']}:{db_config['port']}/{db_name}"
        )
        
        # Création de l'engine
        engine = create_engine(db_url)
        
        # Export du DataFrame
        df.to_sql(table_name, engine, if_exists=if_exists, index=False)
        
        print(f"✅ Table '{table_name}' exportée avec succès ({len(df)} lignes)")
        return True
        
    except Exception as e:
        print(f"❌ Erreur lors de l'export de '{table_name}': {e}")
        return False
    
    finally:
        if 'engine' in locals():
            engine.dispose()

print("Fonction d'export vers PostgreSQL définie.")
print("\nPour exporter les données, utilisez:")
print("export_dataframe_to_postgres(df_accidents_final, 'accidents', db_conf, db_conf['db_accm'])")